# 🧠 ML Process Pipeline
This notebook walks through the entire Machine Learning pipeline from raw data preprocessing to evaluating the final models and generating SHAP explanations.

### 1. Load and inspect data
We start by loading the dataset. We're using the dataset saved from `prepare_data.py`.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, roc_curve, auc, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Load dataset
df = pd.read_csv('../models/diabetes_data.csv')
df.head()

### 2. Preprocessing (zero handling, KNN imputation, clipping)
*Note: Since we are using `models/diabetes_data.csv`, this was already done in `prepare_data.py`. If you want to see the raw preprocessing logic:*

In [ ]:
# Zeros were replaced with NaN
# df['Glucose'] = df['Glucose'].replace(0, np.nan)
# KNN Imputer was applied to fill NaNs based on 5 nearest neighbors
# df[cols] = KNNImputer(n_neighbors=5).fit_transform(df[cols])
# Values were clipped to biological maximums/minimums
# df['BMI'] = df['BMI'].clip(14, 67)
print('Preprocessing previously applied to models/diabetes_data.csv')

### 3. Feature engineering
These 3 new features provide our models with composite risk indicators.

In [ ]:
# 1. GlucoseBMI: Combined metabolic risk score
# df['GlucoseBMI'] = (df['Glucose'] * df['BMI']) / 100

# 2. AgeInsulinRisk: Age-weighted insulin inefficiency
# df['AgeInsulinRisk'] = df['Age'] * (1 / (df['Insulin'] + 1)) * 100

# 3. MetabolicScore: Composite baseline risk
# df['MetabolicScore'] = (df['Glucose'] / 100) + (df['BMI'] / 10) + (df['Age'] / 50)
print('Engineered features are already present in the DataFrame:')
df[['GlucoseBMI', 'AgeInsulinRisk', 'MetabolicScore']].head()

### 4. Scaling with StandardScaler
Models like Gradient Boosting are somewhat robust to scale, but scaling helps the models converge and levels the playing field for feature importance analysis.

In [ ]:
# Separate features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Features scaled.')

### 5. SMOTE for class balance
Because the dataset has roughly 65% non-diabetic and 35% diabetic cases, we use Synthetic Minority Over-sampling Technique (SMOTE) to create synthetic examples of the minority class. This prevents the model from being biased toward predicting 'No Diabetes'.

In [ ]:
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_scaled, y)

print(f'Before SMOTE: {np.bincount(y)}')
print(f'After SMOTE: {np.bincount(y_res)}')

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)

### 6. Train Gradient Boosting
We train a Gradient Boosting model with carefully chosen hyperparameters. This is our primary model (60% weight).

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)
gb_model.fit(X_train, y_train)
print('Gradient Boosting Trained.')

### 7. Train XGBoost
We train an XGBoost model. This acts as our secondary model (40% weight) to smooth out variance via ensembling.

In [ ]:
xgb_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.08, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
print('XGBoost Trained.')

### 8. Evaluate both with confusion matrix + ROC curve + F1
We use AUC-ROC (to measure distinction between classes) and F1 Score (harmonic mean of precision and recall) as our primary metrics.

In [ ]:
gb_preds = gb_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, gb_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('GB Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, gb_model.predict_proba(X_test)[:, 1])
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f'GB AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.legend(loc='lower right')
plt.title('Receiver Operating Characteristic')
plt.show()

print('GB F1 Score:', f1_score(y_test, gb_preds))
print('XGB F1 Score:', f1_score(y_test, xgb_preds))

### 9. SHAP summary plot + waterfall plot for one patient
SHAP (SHapley Additive exPlanations) breaks down exactly *why* a model made a specific prediction by calculating each feature's contribution.

In [ ]:
explainer = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_test)

# Summary Plot
shap.summary_plot(shap_values, X_test, feature_names=X.columns)

# For waterfall plot, we look at the first test patient
explanation = shap.Explanation(values=shap_values[0], base_values=explainer.expected_value[0], data=X_test[0], feature_names=X.columns)
shap.waterfall_plot(explanation)

### 10. End-to-end prediction demo with sample inputs
Here we run a sample patient through the ensemble pipeline.

In [ ]:
sample_patient = pd.DataFrame([{
    'Pregnancies': 2,
    'Glucose': 130,
    'BloodPressure': 75,
    'SkinThickness': 25,
    'Insulin': 100,
    'BMI': 28.5,
    'DiabetesPedigreeFunction': 0.45,
    'Age': 35,
    'GlucoseBMI': (130 * 28.5) / 100,
    'AgeInsulinRisk': 35 * (1 / (100 + 1)) * 100,
    'MetabolicScore': (130 / 100) + (28.5 / 10) + (35 / 50)
}])

# Scale
scaled_patient = scaler.transform(sample_patient)

# Predict probabilities
gb_prob = gb_model.predict_proba(scaled_patient)[:, 1][0]
xgb_prob = xgb_model.predict_proba(scaled_patient)[:, 1][0]
final_prob = 0.6 * gb_prob + 0.4 * xgb_prob

print(f'Gradient Boosting Probability: {gb_prob:.2%}')
print(f'XGBoost Probability: {xgb_prob:.2%}')
print(f'Ensemble Final Risk: {final_prob:.2%}')